# Session 31: LLMs, Generative AI, RAG & Agentic AI
### Duration: ~2–2.5 Hours · **Final session of the course**

---

## Session Goal

In **Session 30** you trained an **LSTM** on text. In production today, most language AI uses **Large Language Models (LLMs)** built on **Transformers**.

This session is a **hands-on** introduction to the modern stack:

- **LLMs** via **Core42** (OpenAI-compatible API)
- **RAG** with **Chroma** vector database and **default embeddings**
- **Agentic AI** — models that plan, call tools, and act in loops

> **Prerequisites:** Sessions 24–25 (neural nets), Session 30 (text/sequences).

> **Setup:** Copy `.env.example` → `.env` and set your Core42 credentials. Install deps in the first code cell.

## What Students Will Learn

- The path from RNN/LSTM → Transformer → LLM
- Tokens, context windows, and prompting patterns
- RAG pipeline: chunk → embed (Chroma default) → retrieve → augment prompt → **Core42 generate**
- Agent loop: goal → plan → tool call → observe → repeat
- Risks: hallucination, privacy, cost, and when **not** to use Gen AI


---
## 1. From Session 30 to Modern NLP

| Era | Architecture | Memory / strength | Typical scale |
|-----|--------------|-------------------|---------------|
| Classical | TF-IDF + logistic regression | Bag of words | Small |
| Session 30 | LSTM | Sequential hidden state | Small–medium |
| **Today** | **Transformer (attention)** | All tokens attend to each other | **Billions of params** |

### What is an LLM?

A **Large Language Model** is a Transformer **pretrained** on huge text corpora to predict the **next token**. After training it can answer questions, summarize, write code, and follow instructions.

We call **Core42** (`https://api.core42.ai/v1`) using the **OpenAI Python SDK** — same `chat.completions` interface, different provider.


In [10]:
# Run once if imports fail:
# %pip install chromadb openai python-dotenv

sample = "Machine learning turns data into decisions."
tokens = sample.lower().replace(".", "").split()

print("Text:", sample)
print("Token count (whitespace split):", len(tokens))
print("Real LLMs use subword tokenizers (BPE) — token count differs.")

Text: Machine learning turns data into decisions.
Token count (whitespace split): 6
Real LLMs use subword tokenizers (BPE) — token count differs.


---
## 2. Environment & Core42 LLM Client

Credentials live in **`.env`** (never commit to git):

| Variable | Purpose |
|----------|--------|
| `CORE42_CHAT_ENDPOINT` | API base URL |
| `CORE42_CHAT_API_KEY` | Your API key |
| `CORE42_CHAT_MODEL_NAME` | Model id (e.g. `gpt-5`) |

> **Note:** Core42 uses `max_completion_tokens` (not `max_tokens`) in API calls.


In [11]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

# Load .env from project root (same folder as this notebook)
load_dotenv(Path(".env"))

CORE42_CHAT_ENDPOINT = os.getenv("CORE42_CHAT_ENDPOINT", "https://api.core42.ai/v1")
CORE42_CHAT_API_KEY = os.getenv("CORE42_CHAT_API_KEY")
CORE42_CHAT_MODEL_NAME = os.getenv("CORE42_CHAT_MODEL_NAME", "gpt-5")

if not CORE42_CHAT_API_KEY:
    raise ValueError(
        "CORE42_CHAT_API_KEY missing. Copy .env.example to .env and set your key."
    )

llm_client = OpenAI(
    api_key=CORE42_CHAT_API_KEY,
    base_url=CORE42_CHAT_ENDPOINT,
)

print("Endpoint:", CORE42_CHAT_ENDPOINT)
print("Model:", CORE42_CHAT_MODEL_NAME)
print("Client ready.")

Endpoint: https://api.core42.ai/v1
Model: gpt-5
Client ready.


In [12]:
def build_chat_prompt(system: str, user: str, context: str = "") -> str:
    """Readable prompt preview for teaching."""
    parts = [f"System: {system}"]
    if context.strip():
        parts.append(f"Context:\n{context.strip()}")
    parts.append(f"User: {user}")
    parts.append("Assistant:")
    return "\n\n".join(parts)


def chat_completion(system: str, user: str, max_completion_tokens: int = 10000) -> str:
    """Call Core42 chat API (OpenAI-compatible)."""
    response = llm_client.chat.completions.create(
        model=CORE42_CHAT_MODEL_NAME,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        max_completion_tokens=max_completion_tokens,
    )
    print(response)
    return response.choices[0].message.content


# Quick smoke test (no RAG yet)
reply = chat_completion(
    system="You are a concise Python tutor.",
    user="In one sentence, what is a pandas DataFrame?",
    max_completion_tokens=10000,
)
print(reply)

ChatCompletion(id='chatcmpl-Dvw2QfZHN6I7BqjZqObBOMhQ8Std0', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='A pandas DataFrame is a two-dimensional, labeled, tabular data structure in Python with columns that can hold different data types, similar to a spreadsheet or SQL table.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None), content_filter_results={'hate': {'filtered': False, 'severity': 'safe'}, 'protected_material_code': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}})], created=1782700166, model='gpt-5-2025-08-07', object='chat.completion', moderation=None, service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=172, prompt_tokens=28, total_tokens=200, completion_tokens_details=CompletionTokensDetail

---
## 3. RAG with Chroma (Default Embeddings)

### RAG pipeline

```text
User question
     |
     v
[1] RETRIEVE — Chroma vector search (default embedding model)
     |
     v
[2] AUGMENT — add retrieved chunks to the prompt
     |
     v
[3] GENERATE — Core42 LLM answers from context
```

**Chroma default embedding:** `all-MiniLM-L6-v2` (auto-downloaded on first run, ~80 MB). No separate embedding API key needed.

Data is persisted under `session31_chroma/` (gitignored).


uiuhojojojoijojoooopkp uiuhojojojoijojoooopkp uiuhojojojoijojoooopkp uiuhojojojoijojoooopkp uiuhojojojoijojoooopkp uiuhojojojoijojoooopkp the car is blue uiuhojojojoijojoooopkp uiuhojojojoijojoooopkp uiuhojojojoijojoooopkp uiuhojojojoijojoooopkp uiuhojojojoijojoooopkp uiuhojojojoijojoooopkp uiuhojojojoijojoooopkp

In [13]:
import chromadb
from chromadb.utils import embedding_functions

CHROMA_PATH = "session31_chroma"
COLLECTION_NAME = "course_notes"

DOCUMENTS = [
    "Session 16 covers data preprocessing: imputation, scaling, one-hot encoding, and sklearn Pipelines.",
    "Session 17 introduces supervised learning with logistic regression and random forest on the penguins dataset.",
    "Session 21 teaches hyperparameter tuning with GridSearchCV and RandomizedSearchCV.",
    "Session 23 is an end-to-end ML capstone using IBM Telco customer churn data.",
    "Session 27 builds convolutional neural networks in PyTorch for image classification.",
    "Session 29 is a transfer learning capstone with MobileNetV2 on CIFAR-10 animal classes.",
    "Session 30 covers RNN and LSTM for sequences and compares to TF-IDF logistic regression.",
    "Session 31 covers LLMs, RAG with Chroma, and agentic AI patterns.",
]


def chunk_text(text: str, max_words: int = 40) -> list[str]:
    words = text.split()
    if len(words) <= max_words:
        return [text]
    return [" ".join(words[i : i + max_words]) for i in range(0, len(words), max_words)]


CHUNKS: list[str] = []
for doc in DOCUMENTS:
    CHUNKS.extend(chunk_text(doc))

# Chroma built-in default embedding function
default_ef = embedding_functions.DefaultEmbeddingFunction()

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

# Fresh ingest each run (teaching demo — production would upsert incrementally)
try:
    chroma_client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    embedding_function=default_ef,
)

collection.add(
    documents=CHUNKS,
    ids=[f"chunk_{i}" for i in range(len(CHUNKS))],
)

print(f"Indexed {collection.count()} chunks in Chroma at '{CHROMA_PATH}'")

Indexed 8 chunks in Chroma at 'session31_chroma'


In [14]:
def retrieve(query: str, top_k: int = 3) -> list[tuple[float, str]]:
    """Return (distance, document) pairs from Chroma. Lower distance = more similar."""
    results = collection.query(query_texts=[query], n_results=top_k)
    docs = results["documents"][0]
    distances = results["distances"][0]
    return list(zip(distances, docs))


query = "How do we tune hyperparameters in this course?"
hits = retrieve(query, top_k=3)
print(f"Query: {query}\n")
for dist, chunk in hits:
    print(f"  distance={dist:.4f} | {chunk}")

Query: How do we tune hyperparameters in this course?

  distance=0.7044 | Session 21 teaches hyperparameter tuning with GridSearchCV and RandomizedSearchCV.
  distance=1.4972 | Session 16 covers data preprocessing: imputation, scaling, one-hot encoding, and sklearn Pipelines.
  distance=1.5643 | Session 17 introduces supervised learning with logistic regression and random forest on the penguins dataset.


In [15]:
def rag_answer(question: str, top_k: int = 3) -> dict:
    hits = retrieve(question, top_k=top_k)
    chunks = [text for _, text in hits]
    context = "\n".join(f"- {c}" for c in chunks)

    system = (
        "You are a course teaching assistant. "
        "Answer using ONLY the provided context. "
        "If the context does not contain the answer, say 'I don't know.' "
        "Cite the session number when possible."
    )
    user = f"Context:\n{context}\n\nQuestion: {question}"

    answer = chat_completion(system=system, user=user, max_completion_tokens=10000)
    prompt_preview = build_chat_prompt(system=system, user=question, context=context)

    return {
        "question": question,
        "retrieved": chunks,
        "distances": [d for d, _ in hits],
        "prompt_preview": prompt_preview,
        "answer": answer,
    }


result = rag_answer("Which session uses GridSearchCV?")
print("Retrieved chunks:")
for c in result["retrieved"]:
    print(" ", c)
print("\nLLM answer:")
print(result["answer"])

ChatCompletion(id='chatcmpl-Dvw2UloMWQq8QGTQEsJsLTv1VGIHv', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Session 21.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None), content_filter_results={'hate': {'filtered': False, 'severity': 'safe'}, 'protected_material_code': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}})], created=1782700170, model='gpt-5-2025-08-07', object='chat.completion', moderation=None, service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=270, prompt_tokens=118, total_tokens=388, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=256, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0,

### RAG vs fine-tuning

| Approach | Best when | Trade-off |
|----------|-----------|----------|
| **RAG** | Facts change often, private docs, cite sources | Retrieval + embedding latency |
| **Fine-tuning** | Style, format, domain jargon | Expensive; may still hallucinate facts |
| **Both** | Enterprise assistants | Common in production |


---
## 4. Agentic AI — Models That Use Tools

An **agent** runs a **loop**: plan → choose tool → observe → repeat until the goal is met.

Below: a small agent that calls **`search_course`** (Chroma RAG) and **`calculator`**, then asks **Core42** to write the final answer from tool outputs.


In [16]:
import re


def tool_search_course(query: str) -> str:
    hits = retrieve(query, top_k=2)
    if not hits:
        return "No matching course note found."
    return " | ".join(text for _, text in hits)


def tool_calculator(expression: str) -> str:
    if not re.fullmatch(r"[0-9+\-*/().\s]+", expression):
        return "Error: only numbers and + - * / ( ) allowed"
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"


TOOLS = {
    "search_course": tool_search_course,
    "calculator": tool_calculator,
}


def run_agent(goal: str, max_steps: int = 4) -> str:
    """Simple rule-based planner + Core42 synthesizer (teaching demo)."""
    observations: list[str] = []
    steps: list[tuple[str, str]] = []

    if "grid" in goal.lower() or "hyperparameter" in goal.lower():
        steps.append(("search_course", "GridSearchCV hyperparameter tuning session"))
    if re.search(r"\d+\s*[+\-*/]\s*\d+", goal):
        expr = re.search(r"(\d+\s*[+\-*/]\s*\d+)", goal).group(1)
        steps.append(("calculator", expr))
    if not steps:
        steps.append(("search_course", goal))

    print(f"Goal: {goal}\n")
    for i, (tool_name, tool_input) in enumerate(steps[:max_steps], 1):
        print(f"Step {i} — Action: {tool_name}({tool_input!r})")
        obs = TOOLS[tool_name](tool_input)
        print(f"         Observation: {obs}\n")
        observations.append(f"{tool_name}: {obs}")

    tool_context = "\n".join(f"- {o}" for o in observations)
    final = chat_completion(
        system="You are a helpful agent. Answer the user's goal using ONLY the tool observations.",
        user=f"Goal: {goal}\n\nTool observations:\n{tool_context}",
        max_completion_tokens=10000,
    )
    print("Final answer:")
    print(final)
    return final


run_agent("Which session teaches GridSearchCV, and what is 18 * 7?")

Goal: Which session teaches GridSearchCV, and what is 18 * 7?

Step 1 — Action: search_course('GridSearchCV hyperparameter tuning session')
         Observation: Session 21 teaches hyperparameter tuning with GridSearchCV and RandomizedSearchCV. | Session 16 covers data preprocessing: imputation, scaling, one-hot encoding, and sklearn Pipelines.

Step 2 — Action: calculator('18 * 7')
         Observation: 126

ChatCompletion(id='chatcmpl-Dvw2XBrHFXAa8gJ3tK5DsbQymnYWM', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='- Session 21 teaches GridSearchCV.\n- 18 × 7 = 126.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None), content_filter_results={'hate': {'filtered': False, 'severity': 'safe'}, 'protected_material_code': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': Fals

'- Session 21 teaches GridSearchCV.\n- 18 × 7 = 126.'

---
## 5. Responsible Use & When Not to Use LLMs

| Risk | Mitigation |
|------|------------|
| **Hallucination** | RAG + citations + human review |
| **PII leakage** | Never put secrets in prompts; use `.env` for keys |
| **Prompt injection** | Treat user input as untrusted |
| **Cost / latency** | Limit `top_k`, `max_completion_tokens`; cache retrieval |

> **Classical ML for decisions**, **deep learning for perception & sequences**, **LLMs + RAG + agents for language workflows**.


---
## Practice Exercises

1. Add 3 documents to `DOCUMENTS`, re-run Chroma ingest, and test a new question.
2. Change `top_k` from 3 to 6 — does the answer improve or get noisier?
3. Ask a question **not** in the knowledge base — does the model say "I don't know"?
4. Add a `tool_session_list` that returns all `DOCUMENTS` titles.
5. Swap `DefaultEmbeddingFunction` for `SentenceTransformerEmbeddingFunction` — when would you?

---

## Session 31 Summary

| Topic | Key takeaway |
|-------|--------------|
| **Core42 LLM** | OpenAI SDK + `.env` credentials |
| **Chroma RAG** | Default embeddings + persistent vector store |
| **Gen AI** | Prompt → retrieve → augment → generate |
| **Agents** | Tools + observations → LLM final answer |

### End of Course

Congratulations — you completed **Sessions 1–31**, including modern **LLMs, RAG, and agentic AI**.
